In [153]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [154]:
# Indlæs data
GDP = pd.read_csv("GDP.csv", skiprows=4)
ODR = pd.read_csv("age-dependency-ratio-old.csv")
TFP = pd.read_csv("total-factor-productivity.csv")
HC_DATA = pd.read_excel("pwt110.xlsx", sheet_name="Data")
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']] # Relevante kolonner fra filen HC_DATA (pwt110)
GEO = pd.read_excel("geo_cepii.xls")
OPN = pd.read_csv("Oppenness.csv", skiprows=4)

Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land, hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

In [155]:
# Relevante variabler fra HC_DATA
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Relevante geografiske variable
GEO = GEO[['country', 'landlocked', 'continent', 'lat', 'area']].copy()

opn2002 = OPN[['Country Name', '2002']].copy()

opn2002.columns = ['country', 'TradeOpen2002']

# Beregner BNP per capita ud fra rgdpe og befolkning
HC['gdp_pc'] = HC['rgdpe'] / HC['pop']

# Udtrækker observationsåret 2002 og beholder de variable, der skal bruges som initiale forklarende variable i tværsnittet
base2002 = HC[HC['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()

# Omdøber variablerne, så det tydeligt fremgår, at de er målt i 2002
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# Udtrækker TFP i slutåret 2020
tfp2020 = HC[HC['year'] == 2020][['country','ctfp']].copy()

# Omdøber TFP-variablen for 2020
tfp2020.columns = ['country','TFP2020']

# Merger 2002- og 2020-data, så hvert land får både initial og slut-TFP
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')

# Konstruerer long-difference outcome som ændringen i log(TFP) fra 2002 til 2020
Tvaersnit['GrowthTFP'] = np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])

# Udtrækker old dependency ratio i startåret 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()

# Omdøber variablerne, så de matcher merge-strukturen
odr2002.columns = ['country', 'ODR2002']

# Sammenfletter ODR med tværsnittet, så hvert land får sin initiale ODR-værdi
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')

# Merge med tværsnittet
Tvaersnit = pd.merge(Tvaersnit, GEO, on='country', how='left')

Tvaersnit = pd.merge(Tvaersnit, opn2002, on='country', how='left')

# Fjern evt. manglende observationer efter merge
Tvaersnit = Tvaersnit.dropna()

In [156]:
print(Tvaersnit.describe())

          HC2002    TFP2002     GDPpc2002    TFP2020  GrowthTFP    ODR2002  \
count  97.000000  97.000000     97.000000  97.000000  97.000000  97.000000   
mean    2.495553   0.685673  19505.174638   0.644302  -0.025700  12.830334   
std     0.670612   0.287543  18159.820584   0.210707   0.266992   7.836286   
min     1.088122   0.167584   1048.418252   0.200132  -0.515328   1.583193   
25%     2.098030   0.446838   4909.995922   0.475661  -0.222283   6.080334   
50%     2.538953   0.662850  11713.944910   0.622159  -0.057211   8.787122   
75%     2.971450   0.884870  34817.300051   0.821492   0.157685  20.612047   
max     3.583555   1.394810  84592.225755   1.078187   0.942964  28.059470   

       landlocked        lat          area  TradeOpen2002  
count   97.000000  97.000000  9.700000e+01      97.000000  
mean     0.154639  19.456151  1.072344e+06      77.711973  
std      0.363439  28.739689  2.281513e+06      46.988081  
min      0.000000 -44.283330  3.160000e+02      20.447122

# ODR + geo variabler

In [174]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'area',
    'landlocked',
    'TradeOpen2002'
]].dropna().copy()

# Log-variable
df_model['log_area'] = np.log(df_model['area'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 80)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 log(area) + β4 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 80)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): Kun ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + TradeOpen2002
X2 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + log_area
X3 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'log_area']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + landlocked
X4 = sm.add_constant(df_model[['ODR2002', 'TradeOpen2002', 'log_area', 'landlocked']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Tabel
results = summary_col(
    [model1, model2, model3, model4],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)'],
    regressor_order=['ODR2002', 'TradeOpen2002', 'log_area', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 TradeOpen2002 + β3 log(area) + β4 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)        (3)        (4)    
----------------------------------------------------------
ODR2002        -0.0084*** -0.0084*** -0.0080*** -0.0082***
               (0.0030)   (0.0030)   (0.0030)   (0.0030)  
TradeOpen2002             -0.0002    0.0003     0.0003    
                          (0.0004)   (0.0006)   (0.0006)  
log_area                             0.0205     0.0204    
                                     (0.0148)   (0.0150)  
landlocked                                      -0.0320   
                                                (0.0873)  
R-squared      0.0607     0.0619     0.0787     0.0805    
R-squared Adj. 0.0508     0.0420     0.0490     0.0406    
N              97         97         97         97        
R2             0.061      0.062      0.079      0.081     
Standard errors in parentheses.
* p<.1, ** p<

# HC + geo variabler

In [182]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'HC2002',
    'TradeOpen2002',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_area'] = np.log(df_model['area'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 95)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 log(area) + β5 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 95)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + HC2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TradeOpen2002', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=['ODR2002', 'HC2002', 'TradeOpen2002', 'log_area', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 TradeOpen2002 + β4 log(area) + β5 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)        (3)        (4)       (5)   
-------------------------------------------------------------------
ODR2002        -0.0084*** -0.0121*** -0.0131*** -0.0118** -0.0120**
               (0.0030)   (0.0046)   (0.0048)   (0.0053)  (0.0052) 
HC2002                    0.0574     0.0728     0.0575    0.0573   
                          (0.0661)   (0.0713)   (0.0789)  (0.0797) 
TradeOpen2002                        -0.0004    0.0001    0.0001   
                                     (0.0004)   (0.0007)  (0.0007) 
log_area                                        0.0176    0.0176   
                                                (0.0170)  (0.0172) 
landlocked                                                -0.0314  
                                                          (0.0868) 
R-squared      0.0607     0.0693     0.0742

# Log(GDP) + geo variabler 

In [190]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 100)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 log(area) + β5 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 100)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + log_GDPpc2002
X2 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=['ODR2002', 'log_GDPpc2002', 'TradeOpen2002', 'log_area', 'landlocked'],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 log(GDPpc2002) + β3 TradeOpen2002 + β4 log(area) + β5 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)       (3)       (4)       (5)   
-----------------------------------------------------------------
ODR2002        -0.0084*** -0.0020   -0.0013   -0.0007   -0.0001  
               (0.0030)   (0.0041)  (0.0042)  (0.0043)  (0.0043) 
log_GDPpc2002             -0.0632** -0.0707** -0.0723** -0.0831**
                          (0.0319)  (0.0361)  (0.0368)  (0.0373) 
TradeOpen2002                       0.0003    0.0009    0.0010   
                                    (0.0004)  (0.0006)  (0.0006) 
log_area                                      0.0216    0.0216   
                                              (0.0143)  (0.0145) 
landlocked                                              -0.0829  
                                                        (0.0881) 
R-squared      0.0607     0.1017    0.1045    0.1231    0.1

# Regioner + Geo variabler

In [188]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 185)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 log(area) + β11 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 185)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus',
    'Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia',
    'Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia',
    'Spain','Sweden','Switzerland','Ukraine','United Kingdom',
    'Armenia'
]

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan',
    'Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore',
    'Sri Lanka','Taiwan','Tajikistan','Thailand'
]

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa',
    'Sudan','Togo','Tunisia','Zambia','Zimbabwe'
]

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia',
    'Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua',
    'Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay'
]

north_america = ['Canada', 'United States']
oceania = ['Australia', 'Fiji', 'New Zealand']
middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + regionale dummies
X2 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + TradeOpen2002
X3 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=[
        'ODR2002', 'Europe', 'Asia', 'Africa',
        'LatinAmerica', 'NorthAmerica', 'Oceania',
        'MiddleEast', 'TradeOpen2002', 'log_area',
        'landlocked'
    ],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 Europe + β3 Asia + β4 Africa + β5 LatinAmerica + β6 NorthAmerica + β7 Oceania + β8 MiddleEast + β9 TradeOpen2002 + β10 log(area) + β11 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)       (2)      (3)      (4)      (5)   
-------------------------------------------------------------
ODR2002        -0.0084*** -0.0078  -0.0095  -0.0093  -0.0098 
               (0.0030)   (0.0074) (0.0079) (0.0081) (0.0078)
Europe                    -0.0130  0.0168   0.0057   0.0127  
                          (0.0925) (0.1036) (0.1090) (0.1048)
Asia                      0.1137** 0.1346** 0.0876   0.0880  
                          (0.0521) (0.0554) (0.0744) (0.0741)
Africa                    -0.0149  -0.0272  -0.0464  -0.0432 
                          (0.0794) (0.0849) (0.0791) (0.0826)
LatinAmerica              -0.0076  -0.0136  -0.0369  -0.0398 
                          (0.0618) (0.0632) (0.0681) (0.0691)
NorthAmerica              

# HC + log(GDP) + regionale dummies + geo variabler

In [194]:
# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'TradeOpen2002',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Antal lande og observationer
antal_lande = df_model['country'].nunique()
antal_observationer = df_model.shape[0]

# Automatisk print
print("=" * 220)
print("MODEL:")
print("GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 log(area) + β13 landlocked + ε")
print(f"Antal lande: {antal_lande}")
print(f"Antal observationer: {antal_observationer}")
print("=" * 220)

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus',
    'Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia',
    'Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia',
    'Spain','Sweden','Switzerland','Ukraine','United Kingdom',
    'Armenia'
]

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan',
    'Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore',
    'Sri Lanka','Taiwan','Tajikistan','Thailand'
]

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa',
    'Sudan','Togo','Tunisia','Zambia','Zimbabwe'
]

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia',
    'Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua',
    'Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay'
]

north_america = ['Canada', 'United States']
oceania = ['Australia', 'Fiji', 'New Zealand']
middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): + kontroller
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002',
                               'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002',
                               'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'TradeOpen2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002',
                               'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002',
                               'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast',
                               'TradeOpen2002', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    regressor_order=[
        'ODR2002', 'HC2002', 'log_GDPpc2002',
        'Europe', 'Asia', 'Africa', 'LatinAmerica',
        'NorthAmerica', 'Oceania', 'MiddleEast',
        'TradeOpen2002', 'log_area', 'landlocked'
    ],
    drop_omitted=True,
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)

MODEL:
GrowthTFP = α + β1 ODR2002 + β2 HC2002 + β3 log(GDPpc2002) + β4 Europe + β5 Asia + β6 Africa + β7 LatinAmerica + β8 NorthAmerica + β9 Oceania + β10 MiddleEast + β11 TradeOpen2002 + β12 log(area) + β13 landlocked + ε
Antal lande: 91
Antal observationer: 97

                  (1)        (2)        (3)       (4)       (5)    
-------------------------------------------------------------------
ODR2002        -0.0084*** -0.0034    -0.0023   -0.0016   -0.0029   
               (0.0030)   (0.0068)   (0.0081)  (0.0085)  (0.0084)  
HC2002                    0.1487     0.1470    0.1427    0.1676*   
                          (0.0979)   (0.0996)  (0.1006)  (0.0923)  
log_GDPpc2002             -0.1211*** -0.1257** -0.1283** -0.1426***
                          (0.0446)   (0.0502)  (0.0508)  (0.0478)  
Europe                    0.0814     0.0766    0.0669    0.0931    
                          (0.0989)   (0.0990)  (0.1025)  (0.0976)  
Asia                      0.1748**   0.1722**  0.1233   